# Week 4: COVID-19 Data Engineering, Cleaning & Exploratory Data Analysis
### AnalystLab Africa Data Analytics Internship

This notebook automates the process of loading, cleaning, unpivoting, and consolidating the raw COVID-19 global time-series datasets (Confirmed, Deaths, and Recovered cases) from Johns Hopkins University. It also performs Exploratory Data Analysis (EDA), KPI calculations, and data visualization as required by the Week 4 project rubric.

#### Steps performed:
1. **Load wide-format datasets**.
2. **Unpivot (melt) date columns** into rows for database-friendly ingestion.
3. **Merge datasets** on geographical attributes and Date.
4. **Calculate Daily changes** (New Cases, New Deaths, New Recoveries) from cumulative values.
5. **Calculate Active Cases, Recovery Rate, and Mortality Rate** columns.
6. **Explore the Dataset (Task 1 Check):** Check columns, data types, missing values, date ranges, and numeric summaries.
7. **Calculate Key KPIs (Task 2):** Totals, global growth rate, and country comparisons.
8. **Create Visualizations (Task 3):** Global trend analysis, bar charts of high-impact countries, and mortality rate trends.

## 1. Setup & Raw Data Loading

In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set non-interactive backend for matplotlib
import matplotlib
matplotlib.use('Agg')

# Resolve the datasets folder regardless of the current working directory

def resolve_data_dir():
    candidate_dirs = [
        Path.cwd(),
        Path.cwd() / "week 4",
        Path.cwd().parent / "week 4",
        Path("/Users/mac/Documents/Tech_projects/Analyst_lab/week 4"),
    ]

    for base_dir in candidate_dirs:
        data_dir = base_dir / "Datasets"
        if data_dir.exists():
            return data_dir

    raise FileNotFoundError(
        "Could not locate the Datasets folder. Please ensure the notebook is run from the project directory or the week 4 folder."
    )

DATA_DIR = resolve_data_dir()
confirmed_path = os.path.join(DATA_DIR, "time_series_covid19_confirmed_global.csv")
deaths_path = os.path.join(DATA_DIR, "time_series_covid19_deaths_global.csv")
recovered_path = os.path.join(DATA_DIR, "time_series_covid19_recovered_global.csv")

In [5]:
print("Loading raw datasets...")
df_confirmed = pd.read_csv(confirmed_path)
df_deaths = pd.read_csv(deaths_path)
df_recovered = pd.read_csv(recovered_path)

print(f"Confirmed shape: {df_confirmed.shape}")
print(f"Deaths shape: {df_deaths.shape}")
print(f"Recovered shape: {df_recovered.shape}")

Loading raw datasets...
Confirmed shape: (289, 1147)
Deaths shape: (289, 1147)
Recovered shape: (274, 1147)


## 2. Melting Wide Datasets to Long Format
The dates (columns from 1/22/20 to 3/9/23) are unpivoted to create a single row for each date observation.

In [7]:
id_cols = ['Province/State', 'Country/Region', 'Lat', 'Long']
date_cols_conf = [col for col in df_confirmed.columns if col not in id_cols]
date_cols_death = [col for col in df_deaths.columns if col not in id_cols]
date_cols_rec = [col for col in df_recovered.columns if col not in id_cols]

print("Melting datasets...")
melted_confirmed = df_confirmed.melt(id_vars=id_cols, value_vars=date_cols_conf, var_name='Date', value_name='Confirmed')
melted_deaths = df_deaths.melt(id_vars=id_cols, value_vars=date_cols_death, var_name='Date', value_name='Deaths')
melted_recovered = df_recovered.melt(id_vars=id_cols, value_vars=date_cols_rec, var_name='Date', value_name='Recovered')

# Convert Date columns to standard datetime without assuming one exact format
for df in [melted_confirmed, melted_deaths, melted_recovered]:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print("Melting completed.")

Melting datasets...
Melting completed.


/var/folders/tr/jb5rcznx5cx_5ggz3m54mxsm0000gn/T/ipykernel_63849/547705210.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
/var/folders/tr/jb5rcznx5cx_5ggz3m54mxsm0000gn/T/ipykernel_63849/547705210.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
/var/folders/tr/jb5rcznx5cx_5ggz3m54mxsm0000gn/T/ipykernel_63849/547705210.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


## 3. Consolidate and Merge Data
Join Confirmed, Deaths, and Recovered metrics together. To avoid duplicate columns, coordinate columns are dropped from the recovered dataset prior to merging.

In [8]:
print("Merging Confirmed and Deaths...")
merged = pd.merge(melted_confirmed, melted_deaths, on=['Province/State', 'Country/Region', 'Lat', 'Long', 'Date'], how='outer')

print("Merging Recovered...")
melted_recovered_sub = melted_recovered.drop(columns=['Lat', 'Long'])
merged = pd.merge(merged, melted_recovered_sub, on=['Province/State', 'Country/Region', 'Date'], how='outer')

# Fill missing coordinates and null metrics
merged['Confirmed'] = merged['Confirmed'].fillna(0).astype(int)
merged['Deaths'] = merged['Deaths'].fillna(0).astype(int)
merged['Recovered'] = merged['Recovered'].fillna(0).astype(int)

print("Merge consolidated successfully.")

Merging Confirmed and Deaths...
Merging Recovered...
Merge consolidated successfully.


## 4. Aggregate & Create Daily metrics
We aggregate province/state data to create a country-daily summary dataset, and compute dynamic metrics including Daily New Cases, Active Cases, Recovery Rate, and Mortality Rate.

In [9]:
print("Aggregating to Country level...")
country_daily = merged.groupby(['Country/Region', 'Date']).agg({
    'Confirmed': 'sum',
    'Deaths': 'sum',
    'Recovered': 'sum'
}).reset_index()

country_daily = country_daily.sort_values(by=['Country/Region', 'Date'])

# Calculate Daily New cases/deaths/recoveries
country_daily['Daily New Cases'] = country_daily.groupby('Country/Region')['Confirmed'].diff().fillna(country_daily['Confirmed'])
country_daily['Daily New Deaths'] = country_daily.groupby('Country/Region')['Deaths'].diff().fillna(country_daily['Deaths'])
country_daily['Daily New Recovered'] = country_daily.groupby('Country/Region')['Recovered'].diff().fillna(country_daily['Recovered'])

# Clip negative values (reporting corrections) to 0
country_daily['Daily New Cases'] = country_daily['Daily New Cases'].clip(lower=0).astype(int)
country_daily['Daily New Deaths'] = country_daily['Daily New Deaths'].clip(lower=0).astype(int)
country_daily['Daily New Recovered'] = country_daily['Daily New Recovered'].clip(lower=0).astype(int)

# Calculate Active Cases, Recovery Rate, Mortality Rate
country_daily['Active Cases'] = (country_daily['Confirmed'] - country_daily['Deaths'] - country_daily['Recovered']).clip(lower=0).astype(int)
country_daily['Recovery Rate'] = (country_daily['Recovered'] / country_daily['Confirmed']).fillna(0)
country_daily['Mortality Rate'] = (country_daily['Deaths'] / country_daily['Confirmed']).fillna(0)

# Re-attach mean Lat/Long coordinates
coords = merged.groupby('Country/Region').agg({
    'Lat': 'mean',
    'Long': 'mean'
}).reset_index()

country_daily = pd.merge(country_daily, coords, on='Country/Region', how='left')

# Save output
country_daily_path = os.path.join(DATA_DIR, "covid_country_daily.csv")
country_daily.to_csv(country_daily_path, index=False)
print(f"Saved country summary dataset to: {country_daily_path}")

Aggregating to Country level...
Saved country summary dataset to: /Users/mac/Documents/Tech_projects/Analyst_lab/week 4/Datasets/covid_country_daily.csv


In [10]:
print("Calculating province-level daily change metrics...")
merged['Group_Key'] = merged['Country/Region'] + "_" + merged['Province/State'].fillna("National")
merged = merged.sort_values(by=['Group_Key', 'Date'])

merged['Daily New Cases'] = merged.groupby('Group_Key')['Confirmed'].diff().fillna(merged['Confirmed'])
merged['Daily New Deaths'] = merged.groupby('Group_Key')['Deaths'].diff().fillna(merged['Deaths'])
merged['Daily New Recovered'] = merged.groupby('Group_Key')['Recovered'].diff().fillna(merged['Recovered'])

# Clean daily negatives
merged['Daily New Cases'] = merged['Daily New Cases'].clip(lower=0).astype(int)
merged['Daily New Deaths'] = merged['Daily New Deaths'].clip(lower=0).astype(int)
merged['Daily New Recovered'] = merged['Daily New Recovered'].clip(lower=0).astype(int)

# Calculate Active Cases, Recovery Rate, Mortality Rate at province level
merged['Active Cases'] = (merged['Confirmed'] - merged['Deaths'] - merged['Recovered']).clip(lower=0).astype(int)
merged['Recovery Rate'] = (merged['Recovered'] / merged['Confirmed']).fillna(0)
merged['Mortality Rate'] = (merged['Deaths'] / merged['Confirmed']).fillna(0)

merged.drop(columns=['Group_Key'], inplace=True)

# Save output
global_cleaned_path = os.path.join(DATA_DIR, "covid_global_cleaned.csv")
merged.to_csv(global_cleaned_path, index=False)
print(f"Saved detailed global dataset to: {global_cleaned_path}")

Calculating province-level daily change metrics...
Saved detailed global dataset to: /Users/mac/Documents/Tech_projects/Analyst_lab/week 4/Datasets/covid_global_cleaned.csv


## 5. Exploratory Data Analysis (EDA) - Task 1 Check
We inspect the column names, missing values, data types, date fields, and numeric measures.

In [11]:
print("=== Column Names & Data Types ===")
print(country_daily.dtypes)

print("\n=== Missing Values Check ===")
print(country_daily.isnull().sum())

print("\n=== Date Field Exploration ===")
start_date = country_daily['Date'].min()
end_date = country_daily['Date'].max()
total_days = (end_date - start_date).days + 1
print(f"Date Range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} ({total_days} days)")

print("\n=== Numeric Measures Summary Statistics ===")
numeric_measures = ['Confirmed', 'Deaths', 'Recovered', 'Daily New Cases', 'Daily New Deaths', 'Active Cases']
print(country_daily[numeric_measures].describe().round(2))

=== Column Names & Data Types ===
Country/Region                 object
Date                   datetime64[ns]
Confirmed                       int64
Deaths                          int64
Recovered                       int64
Daily New Cases                 int64
Daily New Deaths                int64
Daily New Recovered             int64
Active Cases                    int64
Recovery Rate                 float64
Mortality Rate                float64
Lat                           float64
Long                          float64
dtype: object

=== Missing Values Check ===
Country/Region         0
Date                   0
Confirmed              0
Deaths                 0
Recovered              0
Daily New Cases        0
Daily New Deaths       0
Daily New Recovered    0
Active Cases           0
Recovery Rate          0
Mortality Rate         0
Lat                    0
Long                   0
dtype: int64

=== Date Field Exploration ===
Date Range: 2020-01-22 to 2023-03-09 (1143 days)

=== Nume

## 6. Key KPI Definition & Calculations - Task 2
We calculate the total confirmed cases, total deaths, recovery rate, daily growth rate, cases by country, and death rate trends.

In [12]:
# Filter for the latest date to get cumulative totals
last_date = country_daily['Date'].max()
last_day_df = country_daily[country_daily['Date'] == last_date]

total_confirmed = last_day_df['Confirmed'].sum()
total_deaths = last_day_df['Deaths'].sum()

# Calculate Recovery Rate pre-August 2021 (due to reporting discontinuation)
recovery_date = pd.to_datetime('2021-08-01')
rec_df = country_daily[country_daily['Date'] == recovery_date]
rec_total_confirmed = rec_df['Confirmed'].sum()
rec_total_recovered = rec_df['Recovered'].sum()
recovery_rate_aug2021 = (rec_total_recovered / rec_total_confirmed) * 100

print(f"Total Confirmed Cases: {total_confirmed:,}")
print(f"Total Deaths: {total_deaths:,}")
print(f"Recovery Rate (Pre-August 2021): {recovery_rate_aug2021:.2f}%")

Total Confirmed Cases: 676,570,149
Total Deaths: 6,881,802
Recovery Rate (Pre-August 2021): 65.37%


In [13]:
# Cases by Country (Top 10)
top_countries = last_day_df.groupby('Country/Region')['Confirmed'].sum().sort_values(ascending=False).head(10).reset_index()
print("=== Top 10 Countries by Confirmed Cases ===")
for idx, row in top_countries.iterrows():
    print(f"{idx+1}. {row['Country/Region']}: {row['Confirmed']:,} cases")

=== Top 10 Countries by Confirmed Cases ===
1. US: 103,802,702 cases
2. India: 44,690,738 cases
3. France: 39,866,718 cases
4. Germany: 38,249,060 cases
5. Brazil: 37,076,053 cases
6. Japan: 33,320,438 cases
7. Korea, South: 30,615,522 cases
8. Italy: 25,603,510 cases
9. United Kingdom: 24,658,705 cases
10. Russia: 22,075,858 cases


In [14]:
# Global daily trend for growth rate & death rate
global_daily = country_daily.groupby('Date').agg({
    'Confirmed': 'sum',
    'Deaths': 'sum',
    'Daily New Cases': 'sum',
    'Daily New Deaths': 'sum'
}).reset_index()

global_daily['Daily Growth Rate (%)'] = global_daily['Confirmed'].pct_change() * 100
global_daily['Death Rate Trend (%)'] = (global_daily['Deaths'] / global_daily['Confirmed']) * 100

print("=== Global Daily Growth & Death Rate Trends (Last 5 Days) ===")
print(global_daily[['Date', 'Confirmed', 'Daily New Cases', 'Daily Growth Rate (%)', 'Death Rate Trend (%)']].tail(5))

=== Global Daily Growth & Death Rate Trends (Last 5 Days) ===
           Date  Confirmed  Daily New Cases  Daily Growth Rate (%)  \
1138 2023-03-05  676024901            59988               0.008303   
1139 2023-03-06  676082941            63196               0.008585   
1140 2023-03-07  676213378           130437               0.019293   
1141 2023-03-08  676392824           179446               0.026537   
1142 2023-03-09  676570149           177325               0.026216   

      Death Rate Trend (%)  
1138              1.017381  
1139              1.017348  
1140              1.017288  
1141              1.017232  
1142              1.017160  


## 7. Data Visualization - Task 3
We plot and save the charts representing the pandemic waves, country caseloads, and fatality rate trends.

In [15]:
sns.set_theme(style="darkgrid")

# 1. Plot Daily New Cases Trend (Daily Growth)
plt.figure(figsize=(12, 6))
plt.fill_between(global_daily['Date'], global_daily['Daily New Cases'] / 1e6, color="skyblue", alpha=0.4)
plt.plot(global_daily['Date'], global_daily['Daily New Cases'] / 1e6, color="slateblue", alpha=0.8, linewidth=2)
plt.title("Global Daily New Cases Trend (Pandemic Waves in Millions)", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("New Cases (Millions)", fontsize=12)
plt.savefig("global_daily_new_cases.png", dpi=300, bbox_inches='tight')
plt.close() # Close to prevent execution display issues in background
print("Saved global_daily_new_cases.png")

Saved global_daily_new_cases.png


In [16]:
# 2. Plot Top 10 Countries by Caseload
plt.figure(figsize=(12, 6))
sns.barplot(data=top_countries, x=top_countries['Confirmed'] / 1e6, y="Country/Region", palette="viridis")
plt.title("Top 10 Countries by Confirmed Cases (Millions)", fontsize=14, fontweight='bold')
plt.xlabel("Total Confirmed Cases (Millions)", fontsize=12)
plt.ylabel("Country/Region", fontsize=12)
plt.savefig("top_10_countries_cases.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved top_10_countries_cases.png")

/var/folders/tr/jb5rcznx5cx_5ggz3m54mxsm0000gn/T/ipykernel_63849/2287567335.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_countries, x=top_countries['Confirmed'] / 1e6, y="Country/Region", palette="viridis")


Saved top_10_countries_cases.png


In [17]:
# 3. Plot Death Rate Trend
plt.figure(figsize=(12, 6))
plt.plot(global_daily['Date'], global_daily['Death Rate Trend (%)'], color="crimson", linewidth=2.5)
plt.title("Global Case Fatality Rate (CFR) Trend Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Fatality Rate (%)", fontsize=12)
plt.savefig("global_fatality_rate_trend.png", dpi=300, bbox_inches='tight')
plt.close()
print("Saved global_fatality_rate_trend.png")

Saved global_fatality_rate_trend.png
